# FabStack in Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antoniomaisto224/FabStack/blob/main/FabStack_Quick_Start_Colab.ipynb)

**FabStack** designs semiconductor and microfabrication process flows as physically dimensioned 2D cross-sections:
deposit, spin-coat, lithography, etch, strip, lift-off, bake, bond and flip.

You don't need to install anything. Run the cells from top to bottom (**Runtime → Run all** works too).

| Section | What it gives you |
|---|---|
| **1. Setup** | Downloads FabStack and installs everything it needs (about 1 minute) |
| **2. Desktop app in your browser** | The full FabStack window, streamed into a browser tab |
| **3. Build flows with Python** | Script process flows and render the cross-sections inline |
| **4. Point-and-click builder** | Add steps from dropdowns, without writing code |
| **5. Open and save `.fab` files** | Upload your own processes, and download what you make here |

> Files you make in Colab are deleted when the runtime shuts down. Download anything you want to keep (see section 5).

## 1. Setup

In [ ]:
#@title ▶️ Install FabStack (run once per session)
import os, subprocess, sys

REPO_URL = "https://github.com/antoniomaisto224/FabStack.git"
FABSTACK_DIR = "/content/FabStack"

if not os.path.isdir(FABSTACK_DIR):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, FABSTACK_DIR], check=True)
else:
    subprocess.run(["git", "-C", FABSTACK_DIR, "pull", "-q"], check=False)

# Python packages (numpy and matplotlib are already in Colab)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "PyQt5>=5.15,<6", "python-docx>=1.1,<2", "Pillow>=10,<13"], check=True)

# System libraries Qt needs on Linux
subprocess.run("apt-get -qq update > /dev/null && apt-get -qq install -y "
               "libxkbcommon-x11-0 libxcb-icccm4 libxcb-image0 libxcb-keysyms1 libxcb-randr0 "
               "libxcb-render-util0 libxcb-shape0 libxcb-xinerama0 libxcb-xfixes0 libxcb-cursor0 "
               "libegl1 libgl1 libfontconfig1 libdbus-1-3 > /dev/null",
               shell=True, check=True)

os.makedirs("/content/fabstack_files", exist_ok=True)
os.environ["FABSTACK_DATA_DIR"] = "/content/fabstack_files/.user-data"
if FABSTACK_DIR not in sys.path:
    sys.path.insert(0, FABSTACK_DIR)

print("✅ FabStack is installed. Continue with section 2 (desktop app) or section 3 (Python).")

## 2. Desktop app in your browser

This runs the **full FabStack desktop app** on Colab's machine and streams the window to a browser tab
through a virtual display (noVNC). Every feature is available: menus, dialogs, the process history,
and the exports.

1. Run the cell below. The first run takes about a minute.
2. Click the link it prints to open the app in a new tab.
3. When you save, use `/content/fabstack_files`. Section 5 downloads files from there.

> The stream works best in Chrome or Edge. If the screen stays black, wait a few seconds and reload the tab.

In [ ]:
#@title ▶️ Launch the FabStack desktop app
SCREEN = "1600x900"  #@param ["1280x720", "1600x900", "1920x1080"]
import os, subprocess, sys, time

if not os.path.exists("/usr/bin/Xvfb") or not os.path.isdir("/usr/share/novnc"):
    print("Installing the virtual display (one time)…")
    subprocess.run("apt-get -qq install -y xvfb x11vnc novnc websockify openbox > /dev/null",
                   shell=True, check=True)

def _restart(name, cmd, env=None):
    subprocess.run(["pkill", "-f", name], check=False)
    return subprocess.Popen(cmd, env=env, stdout=subprocess.DEVNULL, stderr=subprocess.STDOUT)

_display_env = dict(os.environ, DISPLAY=":1", QT_QPA_PLATFORM="xcb", XDG_RUNTIME_DIR="/tmp/runtime-root")
os.makedirs("/tmp/runtime-root", mode=0o700, exist_ok=True)

subprocess.run(["pkill", "-f", "fabstack"], check=False)
_restart("Xvfb :1", ["Xvfb", ":1", "-screen", "0", f"{SCREEN}x24"])
time.sleep(2)
_restart("openbox", ["openbox"], env=_display_env)
_restart("x11vnc", ["x11vnc", "-display", ":1", "-nopw", "-forever", "-shared", "-quiet", "-rfbport", "5901"])
_restart("websockify", ["websockify", "--web", "/usr/share/novnc", "6080", "localhost:5901"])

app_log = open("/content/fabstack_files/desktop_app.log", "w")
fabstack_app = subprocess.Popen([sys.executable, "-m", "fabstack"], cwd="/content/FabStack",
                                env=_display_env, stdout=app_log, stderr=subprocess.STDOUT)
time.sleep(6)

if fabstack_app.poll() is not None:
    print("⚠️ FabStack stopped right after starting. Here is its log:\n")
    print(open("/content/fabstack_files/desktop_app.log").read()[-3000:])
else:
    from google.colab.output import eval_js
    from IPython.display import HTML, display
    base = eval_js("google.colab.kernel.proxyPort(6080)")
    url = base.rstrip("/") + "/vnc.html?autoconnect=true&resize=scale&reconnect=true"
    display(HTML(f'<p style="font-size:16px">✅ FabStack is running. '
                 f'<a href="{url}" target="_blank"><b>Open FabStack in a new tab ↗</b></a></p>'))

## 3. Build flows with Python

The same process engine also runs without a window. Each step is a small dictionary, and FabStack draws the
cross-section after every step. Dimensions are physical (µm); display scaling never changes them.

**Step reference**

| `operation` | Main fields |
|---|---|
| `"Deposit"`, `"Spincoating"`, `"Bond"` | `material`, `thickness_um` |
| `"Lithography"` | `material` (the resist), `windows=[(start, end), …]` as fractions of the wafer width (0–1), `invert_lithography` |
| `"Etch"` | `material`, `depth_um`, `etch_mode="anisotropic"` or `"isotropic"` |
| `"Strip material"`, `"Lift-off"` | `material` |
| `"Bake/Annealing"`, `"Clean / Surface prep"`, `"Flip"` | none (Bake takes `material`) |

To change the substrate, use `{"action": "set_substrate", "material": …, "thickness_um": …}`.

In [ ]:
#@title ▶️ Load the FabStack Python helper
import os, sys, io
os.environ.setdefault("QT_QPA_PLATFORM", "offscreen")
os.environ.setdefault("FABSTACK_DATA_DIR", "/content/fabstack_files/.user-data")
if "/content/FabStack" not in sys.path:
    sys.path.insert(0, "/content/FabStack")

from PyQt5.QtWidgets import QApplication, QMessageBox
# There is no screen for pop-up dialogs, so answer them automatically.
QMessageBox.warning = lambda *a, **k: QMessageBox.Ok
QMessageBox.information = lambda *a, **k: QMessageBox.Ok
QMessageBox.question = lambda *a, **k: QMessageBox.Yes
_qt_app = QApplication.instance() or QApplication([])

from fabstack.presentation.main_window import MainWindow
from IPython.display import Image, display
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")  # FabStack selects the Qt backend on import


class FabStack:
    """A headless FabStack project you can drive from Python."""

    def __init__(self, substrate=None, thickness_um=None):
        self.win = MainWindow()
        if substrate:
            self.run([{"action": "set_substrate", "material": substrate,
                       "thickness_um": thickness_um or 525}], show=False)

    # --- building -------------------------------------------------------
    def step(self, operation, material=None, show=True, **fields):
        command = {"action": "operation", "operation": operation, **fields}
        if material:
            command["material"] = material
            self.win._execute_assistant_command({"action": "add_material", "material": material})
        return self.run([command], show=show)

    def run(self, commands, show=True):
        for command in commands:
            ok, message = self.win._execute_assistant_command(command)
            print(("✅ " if ok else "⚠️ ") + message)
            if not ok:
                break
        if show:
            self.show()
        return self

    # --- viewing --------------------------------------------------------
    def _png(self):
        path = "/tmp/_fabstack_view.png"
        self.win._save_cropped_figure(self.win.canvas.fig, path)
        return path

    def show(self, step=None):
        """Show the latest cross-section, or the one after history step `step`."""
        if step is not None:
            self.win.history_list.setCurrentRow(step)
            _qt_app.processEvents()
        display(Image(self._png()))
        if step is not None:
            self.win.history_list.setCurrentRow(len(self.win.history) - 1)
            _qt_app.processEvents()

    def history(self):
        for i, state in enumerate(self.win.history):
            print(f"{i:>3}  {state.get('title', '')}")

    def show_all(self):
        for i, state in enumerate(self.win.history):
            print(f"Step {i}: {state.get('title', '')}")
            self.show(step=i)

    def materials(self, search=""):
        names = sorted({str(m) for group in self.win.material_library_by_cat.values()
                        for m in group if str(m) != "Void"})
        return [n for n in names if search.lower() in n.lower()]

    # --- files ----------------------------------------------------------
    def save(self, path):
        self.win._save_process_to_path(path)
        print(f"💾 Saved {path}")

    def export_png(self, path):
        self.win._save_cropped_figure(self.win.canvas.fig, path)
        print(f"🖼️ Saved {path}")

    @classmethod
    def open(cls, path):
        fab = cls()
        fab.win._load_process_dict(fab.win._read_process_file_data(path))
        _qt_app.processEvents()
        return fab


print("✅ Ready. Create a project with fab = FabStack()")

### Example: pattern a SiO₂ hard mask and etch silicon

In [ ]:
fab = FabStack()   # starts on the default silicon substrate

fab.step("Deposit", "Silicon dioxide", thickness_um=1.0, show=False)
fab.step("Spincoating", "PR_SU-8 Photoresist", thickness_um=5.0, show=False)
fab.step("Lithography", "PR_SU-8 Photoresist", windows=[(0.40, 0.60)], invert_lithography=False, show=False)
fab.step("Etch", "Silicon dioxide", depth_um=1.0, etch_mode="anisotropic", show=False)
fab.step("Strip material", "PR_SU-8 Photoresist", show=False)
fab.step("Etch", "Silicon (Si)", depth_um=20.0, etch_mode="anisotropic")

In [ ]:
fab.history()      # list every step
fab.show(step=3)   # view the stack right after step 3

In [ ]:
fab.materials("nitride")   # search the materials library

Try your own flow: edit the steps below and run the cell.

In [ ]:
my_fab = FabStack()
my_fab.run([
    {"action": "operation", "operation": "Deposit",     "material": "Silicon Nitride (Si3N4)", "thickness_um": 0.5},
    {"action": "operation", "operation": "Spincoating", "material": "PR_SU-8 Photoresist",     "thickness_um": 2.0},
    {"action": "operation", "operation": "Lithography", "material": "PR_SU-8 Photoresist",
     "windows": [(0.2, 0.35), (0.65, 0.8)], "invert_lithography": False},
    {"action": "operation", "operation": "Etch", "material": "Silicon Nitride (Si3N4)",
     "depth_um": 0.5, "etch_mode": "isotropic"},
])

## 4. Point-and-click builder

Choose an operation, a material and the values, then click **Add step**. The `.fab` file this saves
opens in the desktop app.

In [ ]:
#@title ▶️ Open the builder
import ipywidgets as W

builder = FabStack()
_ops = ["Deposit", "Spincoating", "Lithography", "Etch", "Strip material", "Lift-off",
        "Bake/Annealing", "Clean / Surface prep", "Bond", "Flip"]
op = W.Dropdown(options=_ops, description="Operation")
mat = W.Combobox(options=builder.materials(), value="Silicon dioxide", description="Material",
                 ensure_option=False, placeholder="type to search")
thick = W.FloatText(value=1.0, description="Thickness µm")
depth = W.FloatText(value=1.0, description="Depth µm")
mode = W.ToggleButtons(options=["anisotropic", "isotropic"], description="Etch mode")
wins = W.Text(value="0.4-0.6", description="Windows", placeholder="e.g. 0.2-0.3, 0.7-0.8")
invert = W.Checkbox(value=False, description="Invert (negative tone)")
add_btn = W.Button(description="Add step", button_style="primary", icon="plus")
undo_btn = W.Button(description="Start over", icon="refresh")
fname = W.Text(value="my_process", description="File name")
save_btn = W.Button(description="Save .fab + PNG", icon="save")
log, view = W.Output(), W.Output()

def _visible(*_):
    o = op.value
    mat.layout.display = "none" if o in ("Flip", "Clean / Surface prep") else ""
    thick.layout.display = "" if o in ("Deposit", "Spincoating", "Bond") else "none"
    depth.layout.display = mode.layout.display = "" if o == "Etch" else "none"
    wins.layout.display = invert.layout.display = "" if o == "Lithography" else "none"
    if o in ("Spincoating", "Lithography") and "resist" not in mat.value.lower():
        mat.value = "PR_SU-8 Photoresist"
op.observe(_visible, "value"); _visible()

def _windows(text):
    out = []
    for part in text.replace(" ", "").split(","):
        a, b = part.split("-")
        out.append((float(a), float(b)))
    return out

def _redraw():
    view.clear_output(wait=True)
    with view:
        builder.show()

def _add(_):
    fields, o = {}, op.value
    if o in ("Deposit", "Spincoating", "Bond"):
        fields["thickness_um"] = thick.value
    if o == "Etch":
        fields.update(depth_um=depth.value, etch_mode=mode.value)
    with log:
        if o == "Lithography":
            try:
                fields.update(windows=_windows(wins.value), invert_lithography=invert.value)
            except ValueError:
                print("⚠️ Write windows as start-end fractions, for example 0.2-0.3, 0.7-0.8")
                return
        material = None if o in ("Flip", "Clean / Surface prep") else mat.value
        builder.step(o, material, show=False, **fields)
    _redraw()

def _reset(_):
    global builder
    builder = FabStack()
    log.clear_output()
    _redraw()

def _save(_):
    base = "/content/fabstack_files/" + (fname.value.strip() or "my_process")
    with log:
        builder.save(base + ".fab")
        builder.export_png(base + ".png")

add_btn.on_click(_add); undo_btn.on_click(_reset); save_btn.on_click(_save)
display(W.VBox([
    W.HBox([op, mat]), W.HBox([thick, depth]), mode, W.HBox([wins, invert]),
    W.HBox([add_btn, undo_btn]), W.HBox([fname, save_btn]), log, view,
]))
_redraw()

## 5. Open and save `.fab` files

Files made here are saved in `/content/fabstack_files`. You can also browse them in Colab's file panel (📁 on the left).

In [ ]:
#@title ⬆️ Upload a .fab file and view it
from google.colab import files
import shutil

uploaded = files.upload()
for name in uploaded:
    target = f"/content/fabstack_files/{name}"
    shutil.move(name, target)
    print(f"\n📂 {name}")
    opened = FabStack.open(target)
    opened.history()
    opened.show()

In [ ]:
#@title ⬇️ Download everything in /content/fabstack_files as a zip
import shutil
from google.colab import files
archive = shutil.make_archive("/content/fabstack_files_export", "zip", "/content/fabstack_files")
files.download(archive)

---
### Next steps

- For full desktop speed, install FabStack locally: see the [README](https://github.com/antoniomaisto224/FabStack#development-setup).
- The process assistant runs a local language model, so it is not set up in this notebook.
- Found a bug? Open an issue on [GitHub](https://github.com/antoniomaisto224/FabStack/issues).